# Generate Final Dataset Files

This notebook processes the sampled account sequences and generates the final dataset files for machine learning:
- Flat transaction list file
- Account labels (phisher vs normal)

## Input Files:
- `../processed_data/eoa2seq.pkl` - Sampled account sequences
- `../processed_data/data_Dataset.address_to_index` - Address to index mapping
- `../processed_data/phisher_accounts.txt` - List of phisher accounts

## Output Files:
- `../dataset/MulDiGraph/output_transactions.txt` - Flat transaction records
- `../dataset/MulDiGraph/account_tags.pkl` - Account labels (0=normal, 1=phisher)

## Step 1: Import Required Libraries

In [1]:
import pickle as pkl 
from tqdm import tqdm
import pandas as pd
import os

## Step 2: Define Helper Functions

In [2]:
def load_data(file_path):
    """Load data from a pickle file"""
    with open(file_path, 'rb') as f:
        data = pkl.load(f)
    return data

def save_data(data, file_path):
    """Save data to a pickle file"""
    with open(file_path, 'wb') as f:
        pkl.dump(data, f)
    return

def save_txt(data, file_path):
    """Save transaction list to a text file"""
    with open(file_path, 'w') as f:
        for tran in data:
            line = ",".join(map(str, tran))
            f.write(str(line) + '\n')
    return

def load_txt(file_path):
    """Load phisher account list from text file"""
    data = pd.read_csv(file_path, names=["account"])
    return list(data.account.values)

## Step 3: Load Input Data

Load the sampled account sequences, address mappings, and phisher account list.

In [3]:
# Load account sequences
print("Loading account sequences...")
data = load_data("../processed_data/eoa2seq.pkl")
print(f"Number of accounts: {len(data)}")

# Load address to index mapping
print("\nLoading address mappings...")
address_to_index = load_data("../processed_data/data_Dataset.address_to_index")
print(f"Total unique addresses: {len(address_to_index)}")

# Load phisher account list
print("\nLoading phisher account list...")
phisher_account = load_txt("../processed_data/phisher_accounts.txt")
print(f"Number of phisher accounts: {len(phisher_account)}")

Loading account sequences...
Number of accounts: 10960

Loading address mappings...
Number of accounts: 10960

Loading address mappings...
Total unique addresses: 825820

Loading phisher account list...
Number of phisher accounts: 5480
Total unique addresses: 825820

Loading phisher account list...
Number of phisher accounts: 5480


## Step 4: Process Transactions and Create Labels

Convert account sequences into flat transaction list and assign labels to each account.

In [4]:
trans = []
tags = {}
phisher_count = 0

print("\nProcessing transactions and creating labels...")
for eoa, seq in tqdm(data.items(), desc="Processing transactions"):
    # Assign label: 1 for phisher, 0 for normal
    if eoa in phisher_account:
        tags[address_to_index[eoa]] = 1
        phisher_count += 1
    else:
        tags[address_to_index[eoa]] = 0
    
    # Process each transaction in the sequence
    for tx in seq:
        from_addr = eoa
        to_addr = tx[0]
        
        # Swap if transaction is incoming
        if tx[3] == "IN":
            from_addr, to_addr = to_addr, from_addr
        
        # Append transaction: [from_index, to_index, timestamp, value]
        trans.append([
            address_to_index[from_addr],
            address_to_index[to_addr],
            tx[1],  # timestamp
            tx[2]   # value
        ])

# Sort tags by index
tags = dict(sorted(tags.items(), key=lambda x: x[0]))

print(f"\nProcessing completed:")
print(f"Total transactions: {len(trans)}")
print(f"Total labeled accounts: {len(tags)}")
print(f"Phisher accounts: {phisher_count}")
print(f"Normal accounts: {len(tags) - phisher_count}")


Processing transactions and creating labels...


Processing transactions: 100%|██████████| 10960/10960 [00:04<00:00, 2386.20it/s]


Processing completed:
Total transactions: 3345323
Total labeled accounts: 10960
Phisher accounts: 5480
Normal accounts: 5480


## Step 5: Create Output Directory and Save Files

In [5]:
# Create output directory if it doesn't exist
output_dir = "../dataset/MulDiGraph"
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

# Save account tags (labels)
print("\nSaving account tags...")
save_data(tags, f"{output_dir}/account_tags.pkl")
print(f"✓ Saved: account_tags.pkl ({len(tags)} accounts)")

# Save transaction list
print("\nSaving transaction list...")
save_txt(trans, f"{output_dir}/output_transactions.txt")
print(f"✓ Saved: output_transactions.txt ({len(trans)} transactions)")

print("\n" + "="*60)
print("DATASET FILES GENERATION COMPLETED")
print("="*60)
print(f"Output location: {output_dir}/")
print(f"Files created:")
print(f"  - account_tags.pkl: {len(tags)} labeled accounts")
print(f"  - output_transactions.txt: {len(trans)} transactions")
print("="*60)

Output directory: ../dataset/MulDiGraph

Saving account tags...
✓ Saved: account_tags.pkl (10960 accounts)

Saving transaction list...
✓ Saved: output_transactions.txt (3345323 transactions)

DATASET FILES GENERATION COMPLETED
Output location: ../dataset/MulDiGraph/
Files created:
  - account_tags.pkl: 10960 labeled accounts
  - output_transactions.txt: 3345323 transactions
✓ Saved: output_transactions.txt (3345323 transactions)

DATASET FILES GENERATION COMPLETED
Output location: ../dataset/MulDiGraph/
Files created:
  - account_tags.pkl: 10960 labeled accounts
  - output_transactions.txt: 3345323 transactions
